In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# ============================================================
# BLOCK 1 : LOAD TRAJECTORY DATASET
# ============================================================

import pandas as pd
import numpy as np
import torch

CSV_PATH = "/kaggle/input/datasets/garvitpujari/lsm-training-dataset/tracking_dataset.csv"

df = pd.read_csv(CSV_PATH)

print("="*70)
print("Trajectory Dataset Loaded")
print("="*70)

print("Shape :", df.shape)

print("\nColumns\n")
print(df.columns.tolist())

FEATURES = [
    "cx",
    "cy",
    "width",
    "height",
    "vx",
    "vy",
    "speed",
    "heading",
    "acceleration"
]

TARGETS = [
    "cx",
    "cy"
]

print("\nInput Features")
print(FEATURES)

print("\nPrediction Target")
print(TARGETS)

print("\nUnique Videos :", df["video_id"].nunique())

print("Unique Track IDs :", df["track_id"].nunique())

display(df.head())

Trajectory Dataset Loaded
Shape : (20799, 17)

Columns

['video_id', 'frame_id', 'track_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy', 'width', 'height', 'vx', 'vy', 'speed', 'heading', 'acceleration']

Input Features
['cx', 'cy', 'width', 'height', 'vx', 'vy', 'speed', 'heading', 'acceleration']

Prediction Target
['cx', 'cy']

Unique Videos : 20
Unique Track IDs : 123


,video_id,frame_id,track_id,confidence,x1,y1,x2,y2,cx,cy,width,height,vx,vy,speed,heading,acceleration
0,video01,3,1,0.904163,928.097107,559.881409,1173.236816,702.410034,1050.666992,631.145752,245.139709,142.528625,-1.981934,0.039978,1.982337,3.121424,1.782492
1,video01,4,1,0.896066,925.846802,559.630310,1169.954468,701.614258,1047.900635,630.622314,244.107666,141.983948,-2.766357,-0.523438,2.815443,-2.954588,0.833106
2,video01,5,1,0.893714,922.317932,560.483398,1164.443481,701.389648,1043.380737,630.936523,242.125549,140.906250,-4.519897,0.314209,4.530806,3.072187,1.715363
3,video01,6,1,0.891884,919.882690,561.700684,1157.249023,699.817017,1038.565918,630.758850,237.366333,138.116333,-4.814819,-0.177673,4.818096,-3.104708,0.287291
4,video01,7,1,0.892384,916.480042,562.464844,1151.115845,699.043884,1033.797974,630.754395,234.635803,136.579041,-4.767944,-0.004456,4.767946,-3.140658,-0.050150


In [2]:
# ============================================================
# BLOCK 2 : CREATE LSTM SEQUENCES
# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np

SEQUENCE_LENGTH = 20

X = []
y = []

trajectory_info = []

# ------------------------------------------------------------
# Group by complete trajectory
# ------------------------------------------------------------

groups = df.groupby(["video_id", "track_id"])

for (video_id, track_id), group in groups:

    group = group.sort_values("frame_id").reset_index(drop=True)

    data = group[FEATURES].values
    target = group[TARGETS].values

    # Need at least 21 frames
    if len(group) <= SEQUENCE_LENGTH:
        continue

    for i in range(len(group) - SEQUENCE_LENGTH):

        X.append(data[i:i+SEQUENCE_LENGTH])

        y.append(target[i+SEQUENCE_LENGTH])

        trajectory_info.append((video_id, track_id))

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("="*70)
print("Sequences Created")
print("="*70)

print("Input Shape :", X.shape)
print("Target Shape:", y.shape)

print("\nEach sample contains:")
print(f"{SEQUENCE_LENGTH} Frames")
print(f"{len(FEATURES)} Features")

print("\nPrediction Target : Next Frame (cx, cy)")

Sequences Created
Input Shape : (18983, 20, 9)
Target Shape: (18983, 2)

Each sample contains:
20 Frames
9 Features

Prediction Target : Next Frame (cx, cy)


In [3]:
# ============================================================
# BLOCK 3 : TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# ------------------------------------------------------------
# Unique trajectories
# ------------------------------------------------------------

unique_tracks = list(set(trajectory_info))

train_tracks, val_tracks = train_test_split(
    unique_tracks,
    test_size=0.20,
    random_state=42
)

train_tracks = set(train_tracks)
val_tracks = set(val_tracks)

# ------------------------------------------------------------
# Split sequences
# ------------------------------------------------------------

train_idx = []
val_idx = []

for i, traj in enumerate(trajectory_info):

    if traj in train_tracks:
        train_idx.append(i)
    else:
        val_idx.append(i)

X_train = X[train_idx]
X_val = X[val_idx]

y_train = y[train_idx]
y_val = y[val_idx]

print("="*70)
print("TRAIN / VALIDATION SPLIT")
print("="*70)

print("Train Samples :", len(X_train))
print("Validation Samples :", len(X_val))

# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, len(FEATURES))
X_val_2d = X_val.reshape(-1, len(FEATURES))

scaler.fit(X_train_2d)

X_train = scaler.transform(X_train_2d).reshape(
    X_train.shape
)

X_val = scaler.transform(X_val_2d).reshape(
    X_val.shape
)

# ------------------------------------------------------------
# Save Scaler
# ------------------------------------------------------------

joblib.dump(
    scaler,
    "/kaggle/working/lstm_scaler.pkl"
)

print("\nScaler Saved Successfully")

print("/kaggle/working/lstm_scaler.pkl")

print("\nFinal Shapes")

print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)

print("y_train :", y_train.shape)
print("y_val   :", y_val.shape)

TRAIN / VALIDATION SPLIT
Train Samples : 17093
Validation Samples : 1890

Scaler Saved Successfully
/kaggle/working/lstm_scaler.pkl

Final Shapes
X_train : (17093, 20, 9)
X_val   : (1890, 20, 9)
y_train : (17093, 2)
y_val   : (1890, 2)


In [4]:
# ============================================================
# BLOCK 4 : PYTORCH DATASET & DATALOADER
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class DroneTrajectoryDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ------------------------------------------------------------
# Create Dataset
# ------------------------------------------------------------

train_dataset = DroneTrajectoryDataset(X_train, y_train)
val_dataset   = DroneTrajectoryDataset(X_val, y_val)

# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("="*70)
print("DATALOADER CREATED")
print("="*70)

print("Training Samples   :", len(train_dataset))
print("Validation Samples :", len(val_dataset))

print("Training Batches   :", len(train_loader))
print("Validation Batches :", len(val_loader))

sample_X, sample_y = next(iter(train_loader))

print("\nBatch Input Shape  :", sample_X.shape)
print("Batch Target Shape :", sample_y.shape)

print("\nInput Features :", sample_X.shape[-1])
print("Sequence Length :", sample_X.shape[1])

DATALOADER CREATED
Training Samples   : 17093
Validation Samples : 1890
Training Batches   : 268
Validation Batches : 30

Batch Input Shape  : torch.Size([64, 20, 9])
Batch Target Shape : torch.Size([64, 2])

Input Features : 9
Sequence Length : 20


In [5]:
# ============================================================
# BLOCK 5 : LSTM MODEL
# ============================================================

import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", DEVICE)

# ============================================================
# LSTM
# ============================================================

class DroneTrajectoryLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=9,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.regressor = nn.Sequential(

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,2)

        )

    def forward(self,x):

        output,(hidden,cell)=self.lstm(x)

        x=hidden[-1]

        x=self.regressor(x)

        return x

# ============================================================
# MODEL
# ============================================================

model = DroneTrajectoryLSTM().to(DEVICE)

print("="*70)
print(model)
print("="*70)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTotal Parameters :", total_params)
print("Trainable Parameters :", trainable_params)

# ============================================================
# LOSS & OPTIMIZER
# ============================================================

criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

print("\nModel Ready For Training")

Device : cuda
DroneTrajectoryLSTM(
  (lstm): LSTM(9, 128, num_layers=2, batch_first=True, dropout=0.3)
  (regressor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=2, bias=True)
  )
)

Total Parameters : 213666
Trainable Parameters : 213666

Model Ready For Training


In [6]:
# ============================================================
# BLOCK 6 : TRAIN LSTM
# ============================================================

from tqdm import tqdm
import copy
import torch

EPOCHS = 50
PATIENCE = 8

best_loss = float("inf")
patience_counter = 0

train_history = []
val_history = []

scaler = torch.amp.GradScaler("cuda")

print("="*70)
print("TRAINING STARTED")
print("="*70)

for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    running_loss = 0

    loop = tqdm(train_loader)

    for X_batch, y_batch in loop:

        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda"):

            pred = model(X_batch)

            loss = criterion(pred, y_batch)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        loop.set_description(
            f"Epoch {epoch+1}/{EPOCHS}"
        )

        loop.set_postfix(
            train_loss=loss.item()
        )

    train_loss = running_loss / len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    running_val = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            pred = model(X_batch)

            loss = criterion(pred, y_batch)

            running_val += loss.item()

    val_loss = running_val / len(val_loader)

    scheduler.step(val_loss)

    train_history.append(train_loss)
    val_history.append(val_loss)

    print(
        f"\nEpoch [{epoch+1}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.6f} "
        f"| Val Loss: {val_loss:.6f}"
    )

    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if val_loss < best_loss:

        best_loss = val_loss

        torch.save(
            model.state_dict(),
            "/kaggle/working/best_lstm.pth"
        )

        patience_counter = 0

        print("✅ Best Model Saved")

    else:

        patience_counter += 1

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")
        break

print("\nTraining Complete.")

print("Best Validation Loss :", best_loss)

print("\nModel Saved At")

print("/kaggle/working/best_lstm.pth")

TRAINING STARTED


Epoch 1/50: 100%|██████████| 268/268 [00:03<00:00, 67.79it/s, train_loss=1.15e+5] 



Epoch [1/50] | Train Loss: 314210.249636 | Val Loss: 117916.650277
✅ Best Model Saved


Epoch 2/50: 100%|██████████| 268/268 [00:02<00:00, 121.58it/s, train_loss=3.71e+4]



Epoch [2/50] | Train Loss: 37510.887251 | Val Loss: 17337.521919
✅ Best Model Saved


...


Epoch [18/50] | Train Loss: 2928.623757 | Val Loss: 3299.052628


Epoch 19/50: 100%|██████████| 268/268 [00:02<00:00, 122.21it/s, train_loss=704]    



Epoch [19/50] | Train Loss: 2888.698115 | Val Loss: 4049.528351


Epoch 20/50: 100%|██████████| 268/268 [00:02<00:00, 122.38it/s, train_loss=1.95e+3]



Epoch [20/50] | Train Loss: 2798.789559 | Val Loss: 4146.480170

Early stopping triggered.

Training Complete.
Best Validation Loss : 1205.2961733500163

Model Saved At
/kaggle/working/best_lstm.pth


In [7]:
# ============================================================
# LSTM V2
# BLOCK 1 : LOAD DATASET
# ============================================================

import pandas as pd
import numpy as np

CSV_PATH = "/kaggle/input/datasets/garvitpujari/lsm-training-dataset/tracking_dataset.csv"

df = pd.read_csv(CSV_PATH)

print("="*70)
print("LSTM V2 DATASET")
print("="*70)

print("Shape :", df.shape)

FEATURES = [
    "cx",
    "cy",
    "width",
    "height",
    "vx",
    "vy",
    "speed",
    "heading",
    "acceleration"
]

print("\nInput Features")

for f in FEATURES:
    print("-", f)

print("\nUnique Videos :", df.video_id.nunique())
print("Unique Tracks :", df.groupby(["video_id","track_id"]).ngroups)

display(df.head())

LSTM V2 DATASET
Shape : (20799, 17)

Input Features
- cx
- cy
- width
- height
- vx
- vy
- speed
- heading
- acceleration

Unique Videos : 20
Unique Tracks : 123


,video_id,frame_id,track_id,confidence,x1,y1,x2,y2,cx,cy,width,height,vx,vy,speed,heading,acceleration
0,video01,3,1,0.904163,928.097107,559.881409,1173.236816,702.410034,1050.666992,631.145752,245.139709,142.528625,-1.981934,0.039978,1.982337,3.121424,1.782492
1,video01,4,1,0.896066,925.846802,559.630310,1169.954468,701.614258,1047.900635,630.622314,244.107666,141.983948,-2.766357,-0.523438,2.815443,-2.954588,0.833106
2,video01,5,1,0.893714,922.317932,560.483398,1164.443481,701.389648,1043.380737,630.936523,242.125549,140.906250,-4.519897,0.314209,4.530806,3.072187,1.715363
3,video01,6,1,0.891884,919.882690,561.700684,1157.249023,699.817017,1038.565918,630.758850,237.366333,138.116333,-4.814819,-0.177673,4.818096,-3.104708,0.287291
4,video01,7,1,0.892384,916.480042,562.464844,1151.115845,699.043884,1033.797974,630.754395,234.635803,136.579041,-4.767944,-0.004456,4.767946,-3.140658,-0.050150


In [8]:
# ============================================================
# BLOCK 2 : CREATE MULTI-STEP TRAJECTORY DATASET
# ============================================================

import numpy as np

SEQUENCE_LENGTH = 20
PREDICT_FRAMES = 5

FEATURES = [
    "cx",
    "cy",
    "width",
    "height",
    "vx",
    "vy",
    "speed",
    "heading",
    "acceleration"
]

X = []
y = []
trajectory_info = []

groups = df.groupby(["video_id","track_id"])

for (video_id,track_id),group in groups:

    group = group.sort_values("frame_id").reset_index(drop=True)

    feature_data = group[FEATURES].values

    center_data = group[["cx","cy"]].values

    # Need 20 input frames + 5 prediction frames
    if len(group) < SEQUENCE_LENGTH + PREDICT_FRAMES:
        continue

    for i in range(len(group)-SEQUENCE_LENGTH-PREDICT_FRAMES+1):

        # -----------------------------
        # INPUT
        # -----------------------------

        X.append(
            feature_data[
                i:i+SEQUENCE_LENGTH
            ]
        )

        # -----------------------------
        # OUTPUT
        # (cx,cy) of next 5 frames
        # -----------------------------

        future = center_data[
            i+SEQUENCE_LENGTH:
            i+SEQUENCE_LENGTH+PREDICT_FRAMES
        ]

        y.append(
            future.flatten()
        )

        trajectory_info.append(
            (video_id,track_id)
        )

X = np.asarray(X,dtype=np.float32)
y = np.asarray(y,dtype=np.float32)

print("="*70)
print("MULTI STEP DATASET CREATED")
print("="*70)

print("Input Shape :",X.shape)
print("Target Shape:",y.shape)

print("\nSequence Length :",SEQUENCE_LENGTH)
print("Future Frames :",PREDICT_FRAMES)

print("\nInput Features :",len(FEATURES))
print("Output Values :",y.shape[1])

print("\nExample Target")

print(y[0])

MULTI STEP DATASET CREATED
Input Shape : (18689, 20, 9)
Target Shape: (18689, 10)

Sequence Length : 20
Future Frames : 5

Input Features : 9
Output Values : 10

Example Target
[985.40796 605.0972  990.33594 602.15234 993.0823  599.22614 994.1821
 598.2263  998.87585 597.4346 ]


In [9]:
# ============================================================
# BLOCK 3 : TRAIN / VALIDATION SPLIT + NORMALIZATION
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import numpy as np

# ------------------------------------------------------------
# Split by trajectory
# ------------------------------------------------------------

unique_tracks = list(set(trajectory_info))

train_tracks, val_tracks = train_test_split(
    unique_tracks,
    test_size=0.20,
    random_state=42
)

train_tracks = set(train_tracks)

train_idx = []
val_idx = []

for i, traj in enumerate(trajectory_info):

    if traj in train_tracks:
        train_idx.append(i)
    else:
        val_idx.append(i)

X_train = X[train_idx]
X_val = X[val_idx]

y_train = y[train_idx]
y_val = y[val_idx]

print("="*70)
print("TRAIN / VALIDATION SPLIT")
print("="*70)

print("Train :", X_train.shape)
print("Val   :", X_val.shape)

# ------------------------------------------------------------
# Normalize Inputs
# ------------------------------------------------------------

x_scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, X_train.shape[-1])
X_val_flat   = X_val.reshape(-1, X_val.shape[-1])

x_scaler.fit(X_train_flat)

X_train = x_scaler.transform(X_train_flat).reshape(X_train.shape)
X_val   = x_scaler.transform(X_val_flat).reshape(X_val.shape)

# ------------------------------------------------------------
# Normalize Targets
# ------------------------------------------------------------

y_scaler = StandardScaler()

y_scaler.fit(y_train)

y_train = y_scaler.transform(y_train)
y_val   = y_scaler.transform(y_val)

# ------------------------------------------------------------
# Save Scalers
# ------------------------------------------------------------

joblib.dump(
    x_scaler,
    "/kaggle/working/input_scaler.pkl"
)

joblib.dump(
    y_scaler,
    "/kaggle/working/target_scaler.pkl"
)

print("\nInput scaler saved.")
print("Target scaler saved.")

print("\nFinal Shapes")

print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)

print("y_train :", y_train.shape)
print("y_val   :", y_val.shape)

print("\nTarget Dimension :", y_train.shape[1])

TRAIN / VALIDATION SPLIT
Train : (16306, 20, 9)
Val   : (2383, 20, 9)

Input scaler saved.
Target scaler saved.

Final Shapes
X_train : (16306, 20, 9)
X_val   : (2383, 20, 9)
y_train : (16306, 10)
y_val   : (2383, 10)

Target Dimension : 10


In [10]:
# ============================================================
# BLOCK 4 : DATASET & DATALOADER
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", DEVICE)

# ============================================================

class DroneTrajectoryDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

# ============================================================

train_dataset = DroneTrajectoryDataset(
    X_train,
    y_train
)

val_dataset = DroneTrajectoryDataset(
    X_val,
    y_val
)

# ============================================================

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================

print("="*70)
print("DATA LOADER READY")
print("="*70)

print("Train Samples :", len(train_dataset))
print("Validation Samples :", len(val_dataset))

print("Train Batches :", len(train_loader))
print("Validation Batches :", len(val_loader))

x,y = next(iter(train_loader))

print("\nInput Shape :", x.shape)
print("Target Shape:", y.shape)

Device : cuda
DATA LOADER READY
Train Samples : 16306
Validation Samples : 2383
Train Batches : 255
Validation Batches : 38

Input Shape : torch.Size([64, 20, 9])
Target Shape: torch.Size([64, 10])


In [11]:
# ============================================================
# BLOCK 5 : PRODUCTION LSTM V2
# ============================================================

import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================

class DroneTrajectoryLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=9,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.norm = nn.LayerNorm(128)

        self.head = nn.Sequential(

            nn.Linear(128,128),
            nn.ReLU(),

            nn.Dropout(0.30),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Linear(64,10)

        )

    def forward(self,x):

        _, (hidden, _) = self.lstm(x)

        x = hidden[-1]

        x = self.norm(x)

        x = self.head(x)

        return x

# ============================================================

model = DroneTrajectoryLSTM().to(DEVICE)

print("="*70)
print(model)
print("="*70)

total = sum(p.numel() for p in model.parameters())

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTotal Parameters :", total)
print("Trainable Parameters :", trainable)

# ============================================================
# LOSS
# ============================================================

criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=5e-4,

    weight_decay=1e-4

)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=4

)

print("\nModel Ready For Training")

DroneTrajectoryLSTM(
  (lstm): LSTM(9, 128, num_layers=2, batch_first=True, dropout=0.3)
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (head): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=10, bias=True)
  )
)

Total Parameters : 228938
Trainable Parameters : 228938

Model Ready For Training


In [12]:
# ============================================================
# BLOCK 6 : TRAIN LSTM V2
# ============================================================

import os
import pandas as pd
from tqdm import tqdm
import torch

EPOCHS = 50
PATIENCE = 8

SAVE_DIR = "/kaggle/working/lstm_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

best_loss = float("inf")
patience_counter = 0

history = []

scaler = torch.amp.GradScaler("cuda")

print("="*70)
print("TRAINING STARTED")
print("="*70)

for epoch in range(EPOCHS):

    # =====================================================
    # TRAIN
    # =====================================================

    model.train()

    train_loss = 0

    loop = tqdm(train_loader)

    for X_batch, y_batch in loop:

        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda"):

            pred = model(X_batch)

            loss = criterion(pred, y_batch)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        loop.set_description(f"Epoch {epoch+1}/{EPOCHS}")

        loop.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)

    # =====================================================
    # VALIDATION
    # =====================================================

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            pred = model(X_batch)

            loss = criterion(pred, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    history.append({

        "epoch": epoch+1,
        "train_loss": train_loss,
        "val_loss": val_loss

    })

    print(
        f"\nEpoch [{epoch+1}/{EPOCHS}]"
        f" | Train Loss: {train_loss:.6f}"
        f" | Val Loss: {val_loss:.6f}"
    )

    # =====================================================
    # SAVE EVERY EPOCH
    # =====================================================

    torch.save(

        model.state_dict(),

        os.path.join(
            SAVE_DIR,
            f"epoch_{epoch+1}.pth"
        )

    )

    # =====================================================
    # SAVE BEST
    # =====================================================

    if val_loss < best_loss:

        best_loss = val_loss

        torch.save(

            model.state_dict(),

            "/kaggle/working/best_lstm_retrained.pth"

        )

        patience_counter = 0

        print("✅ New Best Model Saved")

    else:

        patience_counter += 1

    # =====================================================
    # EARLY STOPPING
    # =====================================================

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")

        break

# =====================================================
# SAVE HISTORY
# =====================================================

history = pd.DataFrame(history)

history.to_csv(

    "/kaggle/working/training_history.csv",

    index=False

)

print("\n"+"="*70)
print("TRAINING FINISHED")
print("="*70)

print("\nBest Validation Loss :", best_loss)

print("\nSaved Files")

print("Best Model")
print("/kaggle/working/best_lstm_retrained.pth")

print("\nAll Epochs")
print("/kaggle/working/lstm_checkpoints/")

print("\nTraining History")
print("/kaggle/working/training_history.csv")

TRAINING STARTED


Epoch 1/50: 100%|██████████| 255/255 [00:02<00:00, 115.17it/s, loss=0.0659]



Epoch [1/50] | Train Loss: 0.201823 | Val Loss: 0.033174
✅ New Best Model Saved


Epoch 2/50: 100%|██████████| 255/255 [00:02<00:00, 114.73it/s, loss=0.0335]



Epoch [2/50] | Train Loss: 0.052175 | Val Loss: 0.009088
✅ New Best Model Saved


...


Epoch [29/50] | Train Loss: 0.025771 | Val Loss: 0.004427


Epoch 30/50: 100%|██████████| 255/255 [00:02<00:00, 117.16it/s, loss=0.0297]



Epoch [30/50] | Train Loss: 0.025400 | Val Loss: 0.004985


Epoch 31/50: 100%|██████████| 255/255 [00:02<00:00, 118.55it/s, loss=0.025] 



Epoch [31/50] | Train Loss: 0.025622 | Val Loss: 0.006442

Early stopping triggered.

TRAINING FINISHED

Best Validation Loss : 0.004202498981229789

Saved Files
Best Model
/kaggle/working/best_lstm_retrained.pth

All Epochs
/kaggle/working/lstm_checkpoints/

Training History
/kaggle/working/training_history.csv


In [ ]:
print('hi')

Device : cuda


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/garvitpujari/lsm-training-dataset/tracking_dataset.csv'